# Importing dependencies

In [1]:
# Dependencies for the missingness tests and imputation comparison.
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display
from scipy.stats import MonteCarloMethod, chi2_contingency
from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer, SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import make_scorer, recall_score
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler


# Loading CSV

In [2]:
# Run the notebook from the project folder.
data_path = Path("hcvdat0.csv")
raw_df = pd.read_csv(data_path)
raw_df.head()


,Unnamed: 0,Category,Age,Sex,ALB,ALP,ALT,AST,BIL,CHE,CHOL,CREA,GGT,PROT
0,1,0=Blood Donor,32,m,38.5,52.5,7.7,22.1,7.5,6.93,3.23,106.0,12.1,69.0
1,2,0=Blood Donor,32,m,38.5,70.3,18.0,24.7,3.9,11.17,4.80,74.0,15.6,76.5
2,3,0=Blood Donor,32,m,46.9,74.7,36.2,52.6,6.1,8.84,5.20,86.0,33.2,79.3
3,4,0=Blood Donor,32,m,43.2,52.0,30.6,22.6,18.9,7.33,4.74,80.0,33.8,75.7
4,5,0=Blood Donor,32,m,39.2,74.1,32.6,24.8,9.6,9.15,4.32,76.0,29.9,68.7


# Data Cleaning and EDA stage

In [3]:
# Remove the patient ID and clean the category labels.
df = raw_df.drop(columns=["Unnamed: 0"], errors="ignore").copy()
df["Category"] = df["Category"].str.replace(r"^[^=]*=", "", regex=True)

category_codes = {
    "Blood Donor": 0,
    "suspect Blood Donor": 1,
    "Suspected Blood Donor": 1,
    "Hepatitis": 2,
    "Fibrosis": 3,
    "Cirrhosis": 4,
}
df["cat_num"] = df["Category"].map(category_codes).astype("Int64")
df["Category"] = df["Category"].replace(
    {"suspect Blood Donor": "Suspected Blood Donor"}
)
print(f"Rows: {df.shape[0]}, columns: {df.shape[1]}")
df.head()


Rows: 615, columns: 14


,Category,Age,Sex,ALB,ALP,ALT,AST,BIL,CHE,CHOL,CREA,GGT,PROT,cat_num
0,Blood Donor,32,m,38.5,52.5,7.7,22.1,7.5,6.93,3.23,106.0,12.1,69.0,0
1,Blood Donor,32,m,38.5,70.3,18.0,24.7,3.9,11.17,4.80,74.0,15.6,76.5,0
2,Blood Donor,32,m,46.9,74.7,36.2,52.6,6.1,8.84,5.20,86.0,33.2,79.3,0
3,Blood Donor,32,m,43.2,52.0,30.6,22.6,18.9,7.33,4.74,80.0,33.8,75.7,0
4,Blood Donor,32,m,39.2,74.1,32.6,24.8,9.6,9.15,4.32,76.0,29.9,68.7,0


In [4]:
# Show each category, its code and its number of patients.
category_summary = (
    df.groupby(["cat_num", "Category"], dropna=False)
      .size()
      .reset_index(name="Count")
      .sort_values("cat_num")
)
category_summary


,cat_num,Category,Count
0,0,Blood Donor,533
1,1,Suspected Blood Donor,7
2,2,Hepatitis,24
3,3,Fibrosis,21
4,4,Cirrhosis,30


In [5]:
# Count missing values and calculate their percentage in each column.
missing_summary = pd.DataFrame({
    "Missing": df.isna().sum(),
    "Missing (%)": df.isna().mean().mul(100).round(2),
})
display(missing_summary)


,Missing,Missing (%)
Category,0,0.00
Age,0,0.00
Sex,0,0.00
ALB,1,0.16
ALP,18,2.93
ALT,1,0.16
AST,0,0.00
BIL,0,0.00
CHE,0,0.00
CHOL,10,1.63


In [6]:
# Missing percentages within each category, before any imputation.
missing_columns = missing_summary.index[
    missing_summary["Missing"] > 0
].tolist()
missing_by_category = (
    df[missing_columns].isna()
    .groupby(df["Category"])
    .mean().mul(100).round(2)
)
display(missing_by_category)


,ALB,ALP,ALT,CHOL,PROT
Category,,,,,
Blood Donor,0.00,0.00,0.00,1.31,0.00
Cirrhosis,3.33,20.00,0.00,6.67,3.33
Fibrosis,0.00,42.86,0.00,4.76,0.00
Hepatitis,0.00,12.50,4.17,0.00,0.00
Suspected Blood Donor,0.00,0.00,0.00,0.00,0.00


In [7]:
# 1. Set up the analysis.
# Use 10,000 resamples and a fixed seed for reproducible estimates.
category_columns = ["Category", "cat_num"]
number_of_resamples = 10000
random_seed = 42
significance_level = 0.05

# Compare patients only when their disease category is known.
known_category_data = df.loc[df[category_columns].notna().all(axis=1)]

# Test incomplete measurements, excluding the diagnosis and its numeric code.
columns_to_test = []
for column in missing_columns:
    if column not in category_columns:
        columns_to_test.append(column)

# 2. Test each measurement separately.
test_results = []
for column in columns_to_test:
    # isna() labels each value as missing (True) or present (False).
    # crosstab() counts these two groups within each disease category.
    missing_flags = known_category_data[column].isna()
    count_table = pd.crosstab(known_category_data["Category"], missing_flags)

    # Without variation in both groups, an association cannot be tested.
    has_multiple_categories = count_table.shape[0] >= 2
    has_missing_and_present = count_table.shape[1] >= 2
    p_value = np.nan

    if has_multiple_categories and has_missing_and_present:
        # Monte Carlo sampling avoids the large-count chi-square approximation.
        # Restart the fixed seed for each column for reproducibility.
        sampling_method = MonteCarloMethod(
            n_resamples=number_of_resamples,
            rng=np.random.default_rng(random_seed),
        )
        test = chi2_contingency(
            count_table, correction=False, method=sampling_method
        )
        p_value = test.pvalue

    test_results.append({
        "Column": column,
        "Missing": int(df[column].isna().sum()),
        "p_value": p_value,
    })

missingness_results = pd.DataFrame(
    test_results, columns=["Column", "Missing", "p_value"]
)

# 3. Adjust for testing several measurements (Bonferroni correction).
# Multiply each p-value by the number of valid tests; p-values cannot exceed 1.
number_of_tests = missingness_results["p_value"].notna().sum()
missingness_results["adjusted_p"] = (
    missingness_results["p_value"] * number_of_tests
).clip(upper=1)

# 4. Explain each adjusted p-value.
# A small p-value suggests association; a large one does NOT prove randomness.
interpretations = []
for adjusted_p in missingness_results["adjusted_p"]:
    if pd.isna(adjusted_p):
        interpretation = "Not testable"
    elif adjusted_p < significance_level:
        interpretation = "Evidence of association with category"
    else:
        interpretation = "No association detected; randomness NOT proven"
    interpretations.append(interpretation)

missingness_results["Interpretation"] = interpretations

# 5. Show the results and the limits of this check.
print(
    "This checks association with Category; it cannot prove randomness "
    "or distinguish MAR from MNAR."
)
missingness_results


This checks association with Category; it cannot prove randomness or distinguish MAR from MNAR.


,Column,Missing,p_value,adjusted_p,Interpretation
0,ALB,1,0.133087,0.665433,No association detected; randomness NOT proven
1,ALP,18,0.000100,0.000500,Evidence of association with category
2,ALT,1,0.083492,0.417458,No association detected; randomness NOT proven
3,CHOL,10,0.217178,1.000000,No association detected; randomness NOT proven
4,PROT,1,0.133087,0.665433,No association detected; randomness NOT proven


### Missing-value findings

Five columns contain missing values: ALB (1), ALP (18), ALT (1), CHOL (10) and PROT (1). The Monte Carlo chi-square tests (10,000 resamples, fixed seed) compared missingness across disease categories, with a Bonferroni adjustment for the five comparisons.

| Column | Missing values | Adjusted p-value | Finding |
|---|---:|---:|---|
| ALB | 1 | 0.6654 | No association detected |
| ALP | 18 | 0.0005 | Evidence of association with disease category |
| ALT | 1 | 0.4175 | No association detected |
| CHOL | 10 | 1.0000 | No association detected |
| PROT | 1 | 0.6654 | No association detected |

**ALP shows the clearest pattern.** It is missing in 42.86% of fibrosis patients (9/21), 20% of cirrhosis patients (6/30) and 12.5% of hepatitis patients (3/24), compared with 0% in both donor groups. Although only 2.9% of ALP values are missing overall, the missingness is concentrated in patients with liver disease. This provides evidence against ALP being missing completely at random (MCAR).

For the other columns, no statistically significant association was detected after adjustment. This does **not** prove their missingness is random. ALB, ALT and PROT each have only one missing value, which limits how much the tests can tell us. These checks only examine disease category and cannot distinguish missing at random (MAR) from missing not at random (MNAR).

**Implication:** dropping rows with missing values could disproportionately remove patients with liver disease, especially fibrosis. Imputation methods should therefore be compared during model validation rather than chosen from the overall missing percentage alone. Replacement values must be learned from training data only; disease category must not be used to fill predictors when it is the outcome being predicted.

The smallest raw Monte Carlo p-value with 10,000 resamples is 1/10,001 (approximately 0.0001). ALP reaches this limit, so its adjusted value of approximately 0.0005 reflects simulation resolution rather than a precise estimate of the underlying p-value. The association remains significant.


# Comparing imputation methods

We cannot prove that median imputation is best by choosing a test that favours it. Instead, compare median, median with missingness flags, and iterative imputation on the **same five validation folds**, using the same two classifiers.

The earlier missingness analysis above uses 615 patients. This section removes the seven suspected donors and uses **608 patients** across Healthy, Hepatitis, Fibrosis and Cirrhosis. No values are filled in the original `df`.

The question is whether a more complex method improves prediction enough to justify using it. Macro F1, balanced accuracy and each class's recall are considered together; no single favourable metric proves superiority.

## 1. Prepare the four-class dataset

In [8]:
# Keep the original dataframe available for the analysis above.
comparison_df = df.loc[
    df["Category"] != "Suspected Blood Donor"
].copy().reset_index(drop=True)
comparison_df["Category"] = comparison_df["Category"].replace({
    "Blood Donor": "Healthy",
})

class_names = ["Healthy", "Hepatitis", "Fibrosis", "Cirrhosis"]
numeric_columns = [
    "Age", "ALB", "ALP", "ALT", "AST", "BIL", "CHE", "CHOL",
    "CREA", "GGT", "PROT",
]
# Explicit predictor names exclude the diagnosis, its code and patient IDs.
features = comparison_df[numeric_columns + ["Sex"]]
target = comparison_df["Category"]

assert len(comparison_df) == 608
assert target.isin(class_names).all()
assert not features.duplicated().any(), "Duplicates need grouped validation."
display(target.value_counts().reindex(class_names).rename("Patients"))


Category
Healthy      533
Hepatitis     24
Fibrosis      21
Cirrhosis     30
Name: Patients, dtype: int64

## 2. Check missing values in the retained patients

In [9]:
# Group percentages use the size of each diagnostic group.
missing_flags = features[numeric_columns].isna()
missing_counts = missing_flags.groupby(target).sum().reindex(class_names)
missing_percent = (
    missing_flags.groupby(target).mean().mul(100).reindex(class_names)
)
missing_features = missing_flags.columns[missing_flags.any()]

display(missing_counts[missing_features])
display(missing_percent[missing_features].round(2))


,ALB,ALP,ALT,CHOL,PROT
Category,,,,,
Healthy,0,0,0,7,0
Hepatitis,0,3,1,0,0
Fibrosis,0,9,0,1,0
Cirrhosis,1,6,0,2,1


,ALB,ALP,ALT,CHOL,PROT
Category,,,,,
Healthy,0.00,0.00,0.00,1.31,0.00
Hepatitis,0.00,12.50,4.17,0.00,0.00
Fibrosis,0.00,42.86,0.00,4.76,0.00
Cirrhosis,3.33,20.00,0.00,6.67,3.33


## 3. Use identical folds and scoring for every method

In [10]:
# Save the splits once so all methods are tested on the same patients.
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
splits = list(cv.split(features, target))
validation_rows = np.concatenate([test for _, test in splits])
assert np.array_equal(np.sort(validation_rows), np.arange(len(target)))

scoring = {
    "Macro F1": "f1_macro",
    "Balanced accuracy": "balanced_accuracy",
}
for category in class_names:
    # A single label gives recall for that diagnosis, including its misses.
    scoring[category + " recall"] = make_scorer(
        recall_score, labels=[category], average="macro", zero_division=0,
    )


## 4. Set the methods and model settings

Median + flags uses the same replacement values as median, but tells the classifier which measurements were missing. Iterative imputation predicts a missing measurement from other numerical predictors. It is **single imputation**, not pooled multiple imputation.

Balanced class weights are the same across all experiments. There is no SMOTE, tuning or diagnosis-based filling.

In [11]:
imputers = {
    "Median": SimpleImputer(strategy="median"),
    "Median + flags": SimpleImputer(strategy="median", add_indicator=True),
    "Iterative": IterativeImputer(
        initial_strategy="median", max_iter=50, tol=0.001,
        min_value=0, random_state=42,
    ),
}
models = {
    "Logistic Regression": LogisticRegression(
        C=1, class_weight="balanced", max_iter=5000, random_state=42,
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=300, min_samples_leaf=2, max_features="sqrt",
        class_weight="balanced", random_state=42, n_jobs=-1,
    ),
}


## 5. Run the comparison

The pipeline fits medians, regression imputations, scaling and Sex encoding on training rows only. `cross_validate()` handles the fold fitting and scoring. Held-out predictions are retained for confusion matrices; no model is fitted twice.

In [12]:
score_tables = []
prediction_tables = []

for model_name, model in models.items():
    for method_name, imputer in imputers.items():
        # Clone each estimator so experiments start with unfitted objects.
        numeric_steps = Pipeline([
            ("imputer", clone(imputer)),
            ("scaler", StandardScaler()),
        ])
        preprocessing = ColumnTransformer([
            ("numeric", numeric_steps, numeric_columns),
            ("sex", OneHotEncoder(handle_unknown="ignore"), ["Sex"]),
        ])
        pipeline = Pipeline([
            ("preprocessing", preprocessing),
            ("model", clone(model)),
        ])
        cv_results = cross_validate(
            pipeline, features, target, cv=splits, scoring=scoring,
            return_estimator=True, error_score="raise",
        )

        fold_scores = pd.DataFrame({
            "Model": model_name, "Imputation": method_name,
            "Fold": range(1, 6),
        })
        for metric in scoring:
            fold_scores[metric] = cv_results["test_" + metric]
        score_tables.append(fold_scores)

        # Each prediction comes from a model that did not train on that row.
        predictions = np.empty(len(target), dtype=object)
        for fitted_model, (_, test) in zip(cv_results["estimator"], splits):
            predictions[test] = fitted_model.predict(features.iloc[test])
        prediction_tables.append(pd.DataFrame({
            "Model": model_name, "Imputation": method_name,
            "Fold": np.zeros(len(target), dtype=int),
            "Actual": target.to_numpy(), "Predicted": predictions,
        }))
        for fold, (_, test) in enumerate(splits, start=1):
            prediction_tables[-1].loc[test, "Fold"] = fold

results = pd.concat(score_tables, ignore_index=True)
held_out_predictions = pd.concat(prediction_tables, ignore_index=True)
assert len(results) == 30
assert len(held_out_predictions) == 6 * len(target)


## 6. Compare mean scores and fold variation

Each number is **mean ± sample standard deviation across five folds**. Standard deviation is not a confidence interval: the training folds overlap. One mistake can change minority-class recall substantially because each fold contains only four or five hepatitis/fibrosis patients.

In [13]:
summary = results.groupby(["Model", "Imputation"])[list(scoring)].agg(
    ["mean", "std"]
)

# Keep the numerical summary, and display a shorter, readable version.
comparison = pd.DataFrame(index=summary.index)
for metric in scoring:
    means = summary[(metric, "mean")].map("{:.3f}".format)
    deviations = summary[(metric, "std")].map("{:.3f}".format)
    comparison[metric] = means + " ± " + deviations

display(comparison[["Macro F1", "Balanced accuracy"]])
display(comparison[[category + " recall" for category in class_names]])


Macro F1 Balanced accuracy
Model               Imputation                                     
Logistic Regression Iterative       0.600 ± 0.058     0.678 ± 0.066
                    Median          0.612 ± 0.061     0.692 ± 0.057
                    Median + flags  0.647 ± 0.020     0.686 ± 0.035
Random Forest       Iterative       0.662 ± 0.128     0.611 ± 0.129
                    Median          0.640 ± 0.149     0.590 ± 0.152
                    Median + flags  0.660 ± 0.155     0.619 ± 0.161

Healthy recall Hepatitis recall  \
Model               Imputation                                       
Logistic Regression Iterative       0.919 ± 0.055    0.680 ± 0.228   
                    Median          0.931 ± 0.040    0.640 ± 0.261   
                    Median + flags  0.974 ± 0.008    0.640 ± 0.297   
Random Forest       Iterative       1.000 ± 0.000    0.370 ± 0.254   
                    Median          1.000 ± 0.000    0.320 ± 0.303   
                    Median + flags  1.000 ± 0.000    0.320 ± 0.268   

                                   Fibrosis recall Cirrhosis recall  
Model               Imputation                                       
Logistic Regression Iterative        0.480 ± 0.254    0.633 ± 0.274  
                    Median           0.530 ± 0.220    0.667 ± 0.312  
                    Median + flags   0.430 ± 0.208    0.700 ± 0.274  
Random Forest       Iterative        0.340 ± 0.290    0.733 ± 0.190  
                    Median           0.340 ± 0.290    0.700 ± 0.217  
                    Median + flags   0.390 ± 0.342    0.767 ± 0.190

## 7. Check differences against median within the same folds

Positive values mean the alternative scored higher than median; negative values favour median. This exposes trade-offs instead of declaring whichever method wins one metric to be universally best.

In [14]:
differences = []
for model_name in models:
    model_results = results.loc[results["Model"] == model_name]
    median_scores = model_results.loc[
        model_results["Imputation"] == "Median"
    ].set_index("Fold")

    for method_name in ["Median + flags", "Iterative"]:
        other_scores = model_results.loc[
            model_results["Imputation"] == method_name
        ].set_index("Fold")
        changes = other_scores[list(scoring)] - median_scores[list(scoring)]
        changes = changes.mean().to_dict()
        changes.update({
            "Model": model_name, "Compared with median": method_name,
        })
        differences.append(changes)

display(pd.DataFrame(differences).set_index([
    "Model", "Compared with median",
]).round(3))


Macro F1  Balanced accuracy  \
Model               Compared with median                                
Logistic Regression Median + flags           0.035             -0.006   
                    Iterative               -0.012             -0.014   
Random Forest       Median + flags           0.019              0.029   
                    Iterative                0.022              0.021   

                                          Healthy recall  Hepatitis recall  \
Model               Compared with median                                     
Logistic Regression Median + flags                 0.043             -0.00   
                    Iterative                     -0.011              0.04   
Random Forest       Median + flags                 0.000             -0.00   
                    Iterative                      0.000              0.05   

                                          Fibrosis recall  Cirrhosis recall  
Model               Compared with median                                     
Logistic Regression Median + flags                  -0.10             0.033  
                    Iterative                       -0.05            -0.033  
Random Forest       Median + flags                   0.05             0.067  
                    Iterative                        0.00             0.033

## 8. Inspect the held-out mistakes

Each patient is counted once in each model/method combination. Rows show actual diagnoses; columns show predictions. These are pooled counts, not average fold scores.

In [15]:
for (model_name, method_name), group in held_out_predictions.groupby(
    ["Model", "Imputation"]
):
    confusion = pd.crosstab(group["Actual"], group["Predicted"]).reindex(
        index=class_names, columns=class_names, fill_value=0,
    )
    assert confusion.to_numpy().sum() == len(target)
    print(f"{model_name} — {method_name}")
    display(confusion)


Logistic Regression — Iterative


Predicted,Healthy,Hepatitis,Fibrosis,Cirrhosis
Actual,,,,
Healthy,490,21,15,7
Hepatitis,2,16,3,3
Fibrosis,2,6,10,3
Cirrhosis,2,3,6,19


Logistic Regression — Median


Predicted,Healthy,Hepatitis,Fibrosis,Cirrhosis
Actual,,,,
Healthy,496,20,10,7
Hepatitis,2,15,5,2
Fibrosis,1,7,11,2
Cirrhosis,1,1,8,20


Logistic Regression — Median + flags


Predicted,Healthy,Hepatitis,Fibrosis,Cirrhosis
Actual,,,,
Healthy,519,5,6,3
Hepatitis,2,15,5,2
Fibrosis,2,7,9,3
Cirrhosis,1,1,7,21


... (2 outputs left out)

Random Forest — Median


Predicted,Healthy,Hepatitis,Fibrosis,Cirrhosis
Actual,,,,
Healthy,533,0,0,0
Hepatitis,12,8,3,1
Fibrosis,7,5,7,2
Cirrhosis,5,1,3,21


Random Forest — Median + flags


Predicted,Healthy,Hepatitis,Fibrosis,Cirrhosis
Actual,,,,
Healthy,533,0,0,0
Hepatitis,11,8,4,1
Fibrosis,7,4,8,2
Cirrhosis,3,1,3,23


## What this comparison can establish

We can justify median imputation as a **simple baseline**, but cannot prove it is the best method for every objective. A method can improve macro F1 while reducing fibrosis recall. Choosing the primary metric after seeing the scores would bias the conclusion.

ALP missingness is concentrated in the disease groups, so adding a flag may encode how tests were collected. This is not automatically target leakage, but performance may change if another clinic measures ALP routinely. Median imputation also does not correct MNAR or guarantee unbiased scientific effect estimates.

This comparison uses one stratified five-fold split design and fixed model settings. It does not establish statistical superiority, performance on independent patients, or the accuracy of genuinely missing laboratory values. It also cannot rule out a better method outside these three candidates.

Peer-reviewed methodological support within the last five years:

- [Sisk et al. (2023), *Imputation and missing indicators for handling missing data in the development and deployment of clinical prediction models*](https://doi.org/10.1177/09622802231165001): choose a strategy consistent with deployment; indicators are conditional on missingness patterns.
- [Shadbahr et al. (2023), *The impact of imputation quality on machine learning classifiers for datasets with missing values*](https://doi.org/10.1038/s43856-023-00356-z): no imputation method consistently produced the best downstream classifier across the studied datasets.

These papers support the comparison design; they do not prove which method wins on this HCV dataset.

## Findings from this run

Median imputation is **not the winner on every metric**. With Logistic Regression:

- Median: macro F1 **0.612**, balanced accuracy **0.692**, fibrosis recall **0.530**.
- Median + flags: macro F1 **0.647**, balanced accuracy **0.686**, fibrosis recall **0.430**.
- Iterative: macro F1 **0.600**, balanced accuracy **0.678**, fibrosis recall **0.480**.

Random Forest + iterative imputation gives the highest macro F1 overall (**0.662**), but its fibrosis recall is **0.340**.

**Decision:** median + Logistic Regression is a defensible simple baseline if balanced accuracy and fibrosis sensitivity matter. It is the highest on those two metrics among these six configurations. The evidence does **not** prove median is universally best; it supports a choice based on the clinical priority. The recommendation is provisional because the same folds were used to compare and select methods, and there is no independent validation.


## Apply the selected median imputation

Create `df_imputed` for the 608 retained patients. Numerical gaps are filled; diagnosis and Sex are unchanged.

This final imputer is fitted on the full cohort after the comparison. For future validation, use the original unfilled data and fit the imputer inside each training fold. Do not cross-validate `df_imputed`.


In [16]:
# Keep the original data unchanged for future validation.
df_imputed = comparison_df.copy()
median_imputer = SimpleImputer(strategy="median")

# Fit on all 608 patients only after the cross-validation above is complete.
df_imputed[numeric_columns] = median_imputer.fit_transform(
    comparison_df[numeric_columns]
)

# Confirm all missing numerical measurements have been filled.
assert not df_imputed[numeric_columns].isna().any().any()
assert len(df_imputed) == 608

display(df_imputed.isna().sum().rename("Missing after imputation"))
display(df_imputed.head())


,Missing after imputation
Category,0
Age,0
Sex,0
ALB,0
ALP,0
ALT,0
AST,0
BIL,0
CHE,0
CHOL,0


,Category,Age,Sex,ALB,ALP,ALT,AST,BIL,CHE,CHOL,CREA,GGT,PROT,cat_num
0,Healthy,32.0,m,38.5,52.5,7.7,22.1,7.5,6.93,3.23,106.0,12.1,69.0,0
1,Healthy,32.0,m,38.5,70.3,18.0,24.7,3.9,11.17,4.80,74.0,15.6,76.5,0
2,Healthy,32.0,m,46.9,74.7,36.2,52.6,6.1,8.84,5.20,86.0,33.2,79.3,0
3,Healthy,32.0,m,43.2,52.0,30.6,22.6,18.9,7.33,4.74,80.0,33.8,75.7,0
4,Healthy,32.0,m,39.2,74.1,32.6,24.8,9.6,9.15,4.32,76.0,29.9,68.7,0


## Save the imputed dataset


In [17]:
# Save without adding the dataframe index as an extra CSV column.
df_imputed.to_csv("hcv_median_imputed.csv", index=False)
